# Explore the datasets (optional)
Downloads both datasets and shows class counts and samples, so you can sanity-check sources before spending GPU time. A CPU runtime is fine for this notebook.

## 1. Setup
In Colab, go to **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.

`REPO_URL` already points at this project on GitHub. If you're working without GitHub, upload the
project folder as a zip and unzip it to `/content/Tyre-defect-detection` instead.

In [ ]:
REPO_URL = "https://github.com/abhinav29890/TyreHealth.git"  # change this if you fork the repo

import os, subprocess
if not os.path.exists("/content/Tyre-defect-detection"):
    subprocess.run(["git", "clone", REPO_URL, "/content/Tyre-defect-detection"], check=True)
%cd /content/Tyre-defect-detection/training
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

### API keys (free)
* **Kaggle**: kaggle.com → Settings → *Create New Token*. In Colab, add secrets (🔑 icon on the left)
  named `KAGGLE_USERNAME` and `KAGGLE_KEY`, and turn on *Notebook access*.
* **Roboflow** is *optional*: it's only needed if you add a Roboflow source to `sources.json`. Use the secret `ROBOFLOW_API_KEY`.

In [ ]:
try:
    from google.colab import userdata
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "ROBOFLOW_API_KEY"):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            if k != "ROBOFLOW_API_KEY":
                print(f"secret {k} not set")
except ImportError:
    print("Not on Colab: set KAGGLE_USERNAME / KAGGLE_KEY / ROBOFLOW_API_KEY as environment variables.")

In [ ]:
!python -m src.prepare_data --task condition --clean
!python -m src.prepare_data --task tread --clean

In [ ]:
import json
for t in ("condition", "tread"):
    print(t, json.load(open(f"data/{t}/summary.json")))

In [ ]:
import random
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

root = Path("data/condition")
classes = [d.name for d in sorted(root.iterdir()) if d.is_dir() and any(d.iterdir())]
fig, axes = plt.subplots(len(classes), 6, figsize=(12, 2.2 * len(classes)), squeeze=False)
for r, c in enumerate(classes):
    files = random.Random(0).sample(sorted((root / c).glob("*.jpg")), k=min(6, len(list((root / c).glob("*.jpg")))))
    for j, ax in enumerate(axes[r]):
        ax.axis("off")
        if j < len(files):
            ax.imshow(Image.open(files[j]))
        if j == 0:
            ax.set_title(c, loc="left")
plt.tight_layout()

In [ ]:
import random
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

root = Path("data/tread")
classes = [d.name for d in sorted(root.iterdir()) if d.is_dir() and any(d.iterdir())]
fig, axes = plt.subplots(len(classes), 6, figsize=(12, 2.2 * len(classes)), squeeze=False)
for r, c in enumerate(classes):
    files = random.Random(0).sample(sorted((root / c).glob("*.jpg")), k=min(6, len(list((root / c).glob("*.jpg")))))
    for j, ax in enumerate(axes[r]):
        ax.axis("off")
        if j < len(files):
            ax.imshow(Image.open(files[j]))
        if j == 0:
            ax.set_title(c, loc="left")
plt.tight_layout()